In [0]:
dbutils.widgets.text("landing_timestamp", "2026-10-03_10:25:18")
dbutils.widgets.text("table_name", "hospitals")
dbutils.widgets.text("merge_keys", "hospital_id")
dbutils.widgets.dropdown("load_type", "FULL", ["FULL", "APPEND", "MERGE"])


landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")


In [0]:
bronze_df = spark.read.table(f"adb_caaresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")


In [0]:
from pyspark.sql.functions import current_timestamp

print("=" * 60)
print(f"  BRONZE → SILVER LOAD")
print("=" * 60)
print(f"  load_type    : {load_type}")
print(f"  table_name   : {table_name}")
print(f"  merge_keys   : {merge_keys}")
print("-" * 60)

# Step 1: Add metadata columns
print("\n[Step 1] Adding metadata columns: insert_timestamp, updated_timestamp...")
updated_df = (bronze_df
    .withColumn("insert_timestamp", current_timestamp())
    .withColumn("updated_timestamp", current_timestamp())
)
print(f"          Columns in source df : {updated_df.columns}")
print(f"          Source record count  : {updated_df.count()}")

target_table = f"adb_caaresync.silver.{table_name}"
print(f"\n[Step 2] Target table          : {target_table}")

# Step 3: Write based on load_type
print(f"\n[Step 3] Executing load_type   : {load_type}")

if load_type == "FULL":
    print("          Strategy            : Overwrite entire target table")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("          ✅ FULL load complete.")

elif load_type == "APPEND":
    print("          Strategy            : Append records to target table")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("          ✅ APPEND load complete.")

elif load_type == "MERGE":
    # Register source as a temp view for SQL MERGE
    updated_df.createOrReplaceTempView("source_view")
    print("          Strategy            : Upsert — insert new, update existing records")
    print("          Source registered as temp view: source_view")

    keys = [k.strip() for k in merge_keys.split(",")]
    join_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])
    print(f"          Merge keys          : {keys}")
    print(f"          Join condition      : {join_condition}")

    # Build update SET clause — all columns except insert_timestamp (preserve original)
    update_cols = [c for c in updated_df.columns if c != "insert_timestamp"]
    update_set  = ", ".join([f"target.{c} = source.{c}" for c in update_cols])
    print(f"          Columns updated     : {update_cols}")
    print(f"          (insert_timestamp excluded from UPDATE — preserving original value)")

    table_exists = spark.catalog.tableExists(target_table)
    print(f"          Target table exists : {table_exists}")

    if table_exists:
        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING source_view          AS source
            ON {join_condition}
            WHEN MATCHED THEN
                UPDATE SET {update_set}
            WHEN NOT MATCHED THEN
                INSERT *
        """
        print(f"\n          Prepared MERGE SQL:")
        print("          " + "-" * 50)
        for line in merge_sql.strip().splitlines():
            print(f"          {line}")
        print("          " + "-" * 50)
        print("\n          Executing MERGE...")
        spark.sql(merge_sql)
        print("          ✅ MERGE complete.")
    else:
        print("          Target table does not exist — creating on first run via overwrite...")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("          ✅ Table created successfully.")

else:
    raise ValueError(f"Unknown load_type: '{load_type}'. Supported values: FULL, APPEND, MERGE.")

print("\n" + "=" * 60)
print("  LOAD COMPLETE")
print("=" * 60)

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))